## Introduction

In this tutorial, We will walk though major aspect of `sorts` by completing an experiment.

This is the simplified simulation architecture of `sorts`:

![diagram](../assets/simplified_simulation_architecture.svg)

As depicted above, a simulation in `sorts` requires the following inputs:

- space objects and their propagations info
- passages and radar stations info
- schedule
- other experiment parameters

We will go over them one by one.

## Space objects and their propagations info

A space object is modeled by the `SpaceObject` class.

Usually, we want to simulate the behavior of more than a single `SpaceObject`.

This can be done either by creating multiple `SpaceObject` instances manually, or by sampling from a population.  
A population represents the entire set of space objects we are interested in studying.

In this tutorial, we will use the latter method.  
Let's create a population.

### Create a population and get a list of space objects

In [1]:
import typing as t
import numpy as np
from astropy.time import Time
from sorts import population

rand_seed = 1234
np.random.seed(rand_seed)
rng = np.random.default_rng(seed=rand_seed)

R_earth = 6371e3
grid_size = (4, 4)  # used a very small grid for demo, normal values are e.g. `(50, 50)`
epoch = Time("2025-01-01 00:00:00")

spobj_pop = population.orbit_grid(
    semi_major_axis_samples=np.linspace(R_earth + 300e3, R_earth + 1000e3, num=grid_size[0]),
    eccentricity_samples=np.array([0]),
    inclination_samples=np.array([0]),
    argument_of_periapsis_samples=np.array([0]),
    longitude_of_ascending_node_samples=np.array([0]),
    mean_anomaly_samples=np.array([0]),
    diameter_samples=10 ** np.linspace(-2, 1, num=grid_size[1]),
    frame="TEME",
    epoch_mjd=t.cast(float, epoch.mjd),
    additional_parameters={"area_to_mass": 0, "m": 0},
    degrees=True,
)
spobj_pop.data["i"] = 75.0
spobj_pop.data["area_to_mass"] = 10 ** (rng.random(len(spobj_pop)) * 4 - 3)
areas = np.pi * (spobj_pop.data["d"] / 2) ** 2
spobj_pop.data["m"] = areas / spobj_pop.data["area_to_mass"]

Then we get a list of space objects from the population.

In [2]:
spobjs = [spobj_pop.get_object(oid) for oid in range(len(spobj_pop))]

A `SpaceObject` encapsulates a object in space who's dynamics is governed by a propagator in time.

For a `SpaceObject` to be useful in a simulation, it needs to be propagated.  
Since propagation can be computationally expensive, we typically propagate it at sparse time steps and then interpolate between them.

Here is how this can be done:

In [3]:
from sorts import InterpolatedPropagation
from sorts import interpolation, propagator

start_time = epoch
end_time = Time("2025-01-02 00:00:00")

interp_props = [
    InterpolatedPropagation.from_space_object(
        space_object=spobj,
        propagator=propagator.Sgp4(
            settings=propagator.Sgp4Settings(out_frame="ITRS", mean_elements_input=True)
        ),
        interpolator_class=interpolation.Legendre8,
        start_time=start_time,
        end_time=end_time,
        time_step=10,
    )
    for spobj in spobjs
]

## Passages and radar stations info

Now, let's look at preparing the passages and radar station information.

We first need to define the radar system we are using, and then proceed to calculate the passages.

There are several built-in radar systems, like EISCAT, EISCAT 3D and NOSTRA.

We will use one of them.

In [4]:
from sorts import radar

duty_cycle = 0.2
coherent_integration_time = 0.04

radar_sys = radar.radars.nostra.gen_nostra(
    frequency=3.2e9,
    antenna_num=10_000,
    antenna_spacing_lambda=0.65,
    antenna_efficiency=0.5,
    antenna_input_power=100,  # W
    thermal_load=1,
    noise_figure_db=0.7,
    amplifier_gain_db=18,
    insertion_loss_db=0.35,
    duty_cycle=duty_cycle,
    t_sky=10.0,
    coherent_integration_time=coherent_integration_time,
    bandwidth_limit_ratio=5,
)

# some further configuration of the radar_sys
tx_station = radar_sys.tx[0]
rx_stations = radar_sys.rx
station_map = {idx: stn for idx, stn in enumerate([tx_station, *rx_stations])}

for idx, stn in station_map.items():
    stn.uid = idx

Then we calculate the passages:

In [5]:
from sorts import passage

start_time_dt64 = t.cast(np.datetime64, start_time.datetime64)

passages_ls = [
    passage.find_simultaneous_passages(
        dt=(interp_prop.times - start_time_dt64) / np.timedelta64(1, "s"),
        space_object=spobj,
        states=interp_prop.states[:3, ...],
        tx_station=tx_station,
        rx_stations=rx_stations,
        epoch=start_time_dt64,
    )
    for spobj, interp_prop in zip(spobjs, interp_props)
]

## Schedule

A schedule captures the time-varying parameters of a radar system, most importantly when and where the radar is pointing.  
It also holds references to time-invariant parameters, such as its power, bandwidth, and other settings.

We use controllers to generate schedules and then merge them for use in the simulation.

There are several built-in controllers, like `FenceScanController`, `TrackerController` and `SparseTrackerController`.

We will use one of them.

In [6]:
from sorts.types import ExperimentDetail
from sorts.controller import SparseTrackerController

time_slice = coherent_integration_time / duty_cycle
control_slice_duration = np.timedelta64(int(time_slice * 1e6), "us")

exp_detail = ExperimentDetail(
    id=0,
    # not used
    coh_int_bandwidth=1.0,
    ipp=1.0,
    pulse_length=1.0,
    duty_cycle=1.0,
    # --
    power=tx_station.power,
    bandwidth=1 / coherent_integration_time,
    noise_temp=rx_stations[0].noise,
    slice_duration=control_slice_duration,
)

tracker_ctrls = [
    SparseTrackerController.from_space_object(
        tx_station=tx_station,
        rx_stations=rx_stations,
        exp_detail=ExperimentDetail(
            id=0,
            # not used
            coh_int_bandwidth=1.0,
            ipp=1.0,
            pulse_length=1.0,
            duty_cycle=1.0,
            # --
            power=tx_station.power,
            bandwidth=1 / coherent_integration_time,
            noise_temp=rx_stations[0].noise,
            slice_duration=control_slice_duration,
        ),
        space_object=spobj,
        epoch=start_time,
        points_per_passage=10,
        interpolator=interp_prop.interpolator,
    )
    for spobj, interp_prop in zip(spobjs, interp_props)
]

Then we create the schedules and merge them.

In [7]:
from sorts import schedule

tracker_ctrl_sch_names = [f"tracker_sch_{idx}" for idx in range(len(tracker_ctrls))]
schedule_db = schedule.ScheduleDb.from_schedule_dataframes(
    [tracker_ctrl.generate(passages) for tracker_ctrl, passages in zip(tracker_ctrls, passages_ls)],
    tracker_ctrl_sch_names,
)
schedule_db.schedule_by_priority()

## Other experiment parameters

In this case, experiment parameters are all captured inside the `exp_detail` in the previous section.

In [8]:
exp_detail

ExperimentDetail(id=0, coh_int_bandwidth=1.0, ipp=1.0, pulse_length=1.0, power=1000000, bandwidth=25.0, duty_cycle=1.0, noise_temp=136.33552708253137, slice_duration=np.timedelta64(199999,'us'))

## Simulation

We now have all the inputs needed for the simulation. Let's create one and then run it.

In [9]:
from sorts.simulation import stx_mrx_simulation

sim_result = stx_mrx_simulation.simulate(
    space_objects=spobjs,
    interpolated_propagations=interp_props,
    passages_list=passages_ls,
    schedule_db=schedule_db,
    station_map=station_map,
    exp_detail_map={exp_detail.id: exp_detail},
)

simulating: 100%|██████████| 16/16 [00:01<00:00, 12.49it/s]


## Simulation result

Simulation result is just a `pandas` `DataFrame`.

In [10]:
import pandas as pd

pd.set_option("display.expand_frame_repr", False)  # config to show all cols without trimming

sim_result

[[                                                  tx_pointing_e  tx_pointing_n  tx_pointing_u  rx_pointing_e  rx_pointing_n  rx_pointing_u       gain_tx       gain_rx           snr       tx_range       rx_range  two_way_range  two_way_range_rate
  exp_num rx_simult_num time                                                                                                                                                                                                                           
  0       0             2025-01-01 18:23:05.454545      -0.135239       0.318004       0.938394      -0.114182       0.339690       0.933581      1.254293  24783.263031  2.363340e-03  324220.143947  324220.143947   6.484403e+05         2663.770594
                        2025-01-01 18:23:05.454545      -0.114182       0.339690       0.933581      -0.135239       0.318004       0.938394  24783.263031      1.254293  2.363340e-03  324220.143947  324220.143947   6.484403e+05         2663.770594
        